# Category C – Transformer-based Natural Language Inference

This notebook implements a transformer-based approach for the Natural Language Inference (NLI) task.

The objective of NLI is to determine the relationship between a **premise** and a **hypothesis**, classifying it into categories such as *entailment*, *contradiction*, or *neutral*.

---

## Approach

This project uses a **hybrid ensemble of two RoBERTa-based models**:

- **Cross-Encoder Model**
  - Takes the premise and hypothesis together as input
  - Captures fine-grained, token-level interactions between the two sentences

- **Bi-Encoder Model**
  - Encodes the premise and hypothesis separately
  - Learns sentence-level semantic representations and compares them

---

## Ensemble Strategy

The final prediction is obtained by combining both models using a weighted average:

final_logits = α · cross_logits + (1 - α) · bi_logits

where \(alpha) is chosen based on development set performance.

- This allows the model to balance detailed interaction (cross-encoder) and general semantic similarity (bi-encoder)

---


In [ ]:
# imports

import os
import random
import json
from typing import Dict, List

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torch.optim import AdamW


from transformers import (
    AutoTokenizer,
    RobertaModel,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

Device: cuda
GPU: Tesla T4


In [2]:
BASE_DIR = os.getcwd()

OUT_DIR = os.path.join(BASE_DIR, "outputs", "category_C")

CROSS_DIR = os.path.join(OUT_DIR, "cross_encoder")
BI_DIR = os.path.join(OUT_DIR, "bi_encoder")
ENSEMBLE_DIR = os.path.join(OUT_DIR, "ensemble")
SUBMISSION_DIR = os.path.join(OUT_DIR, "submissions")

# Create all folders
os.makedirs(CROSS_DIR, exist_ok=True)
os.makedirs(BI_DIR, exist_ok=True)
os.makedirs(ENSEMBLE_DIR, exist_ok=True)
os.makedirs(SUBMISSION_DIR, exist_ok=True)

print("Output directory:", OUT_DIR)

Output directory: /content/outputs/category_C


In [4]:
# paths and loading data

TRAIN_PATH = "data/train.csv"
DEV_PATH = "data/dev.csv"

assert os.path.exists(TRAIN_PATH), "train.csv not found in data/ folder."
assert os.path.exists(DEV_PATH), "dev.csv not found in data/ folder."

MODEL_NAME = "roberta-base"
OUTPUT_DIR = CROSS_DIR
BI_OUTPUT_DIR = BI_DIR
ENSEMBLE_OUTPUT_DIR = ENSEMBLE_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BI_OUTPUT_DIR, exist_ok=True)
os.makedirs(ENSEMBLE_OUTPUT_DIR, exist_ok=True)

print("Found:", TRAIN_PATH, DEV_PATH)
print("Model:", MODEL_NAME)

train_df = pd.read_csv(TRAIN_PATH)
dev_df = pd.read_csv(DEV_PATH)

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)

display(train_df.head())
display(dev_df.head())

Found: data/train.csv data/dev.csv
Model: roberta-base
Train shape: (24432, 3)
Dev shape: (6736, 3)


,premise,hypothesis,label
0,yeah i don't know cut California in half or so...,Yeah. I'm not sure how to make that fit. Maybe...,1
1,actual names will not be used,"For the sake of privacy, actual names are not ...",1
2,The film was directed by Randall Wallace.,The film was directed by Randall Wallace and s...,1
3,"""How d'you know he'll sign me on?""Anse studie...",Anse looked at himself in a cracked mirror.,1
4,In the light of the candles his cheeks looked ...,Drew regarded his best friend and noted that i...,1


,premise,hypothesis,label
0,"By starting at the soft underbelly, the 16,000...","General Nelson A. Miles had 30,000 troops in h...",0
1,"The class had broken into a light sweat, but w...",The class grew more tense as time went on.,1
2,"Samson had his famous haircut here, but he wou...",It was unknown where exactly within the town S...,1
3,A man with a black shirt holds a baby while a ...,A darkly dressed man passes a crying baby to a...,0
4,I know that many of you are interested in addr...,The problems must be addressed,1


In [5]:
# Data prep: detect columns, clean data, label mapping

def guess_column(df: pd.DataFrame, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in cols_lower:
            return cols_lower[cand]
    return None

premise_col = guess_column(train_df, ["premise", "sentence1", "text1", "s1"])
hypothesis_col = guess_column(train_df, ["hypothesis", "sentence2", "text2", "s2"])
label_col = guess_column(train_df, ["label", "gold_label", "class", "y"])

print("Guessed columns:")
print("  premise_col   =", premise_col)
print("  hypothesis_col=", hypothesis_col)
print("  label_col     =", label_col)

if premise_col is None or hypothesis_col is None or label_col is None:
    raise ValueError(f"Could not confidently detect columns. Columns found: {list(train_df.columns)}")

for df in [train_df, dev_df]:
    df[premise_col] = df[premise_col].fillna("").astype(str).str.strip()
    df[hypothesis_col] = df[hypothesis_col].fillna("").astype(str).str.strip()

train_df = train_df[(train_df[premise_col] != "") & (train_df[hypothesis_col] != "")].copy()
dev_df = dev_df[(dev_df[premise_col] != "") & (dev_df[hypothesis_col] != "")].copy()

print("Train label examples:", train_df[label_col].head(10).tolist())
print("Unique train labels:", sorted(train_df[label_col].unique().tolist()))

unique_labels = sorted(train_df[label_col].unique().tolist())
label2id = {lab: i for i, lab in enumerate(unique_labels)}
id2label = {i: lab for lab, i in label2id.items()}

print("label2id:", label2id)

train_df["label_id"] = train_df[label_col].map(label2id)
dev_df["label_id"] = dev_df[label_col].map(label2id)

assert dev_df["label_id"].isna().sum() == 0, "Dev set contains unseen labels."

train_df["label_id"] = train_df["label_id"].astype(int)
dev_df["label_id"] = dev_df["label_id"].astype(int)

num_labels = len(label2id)
print("num_labels:", num_labels)

print("\nTrain label distribution:")
display(train_df[label_col].value_counts())

print("\nDev label distribution:")
display(dev_df[label_col].value_counts())

Guessed columns:
  premise_col   = premise
  hypothesis_col= hypothesis
  label_col     = label
Train label examples: [1, 1, 1, 1, 1, 0, 0, 0, 1, 1]
Unique train labels: [0, 1]
label2id: {0: 0, 1: 1}
num_labels: 2

Train label distribution:


,count
label,
1,12646
0,11783



Dev label distribution:


,count
label,
1,3478
0,3258


In [6]:
# tokenizer and training config

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_GRAD_NORM = 1.0
DROPOUT = 0.2
HIDDEN_DIM_1 = 256
HIDDEN_DIM_2 = 128

print("Model:", MODEL_NAME)
print("Max length:", MAX_LENGTH)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("LR:", LEARNING_RATE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model: roberta-base
Max length: 256
Batch size: 16
Epochs: 2
LR: 2e-05


## Dataset Classes

This section defines the dataset objects used by PyTorch.

- The **cross-encoder dataset** tokenizes the premise and hypothesis together as a sentence pair.
- The **bi-encoder dataset** tokenizes the premise and hypothesis separately so they can be encoded independently.

In [7]:
# dataset classes

class NLIPairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, premise_col: str, hypothesis_col: str, label_col: str, max_length: int):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.premise_col = premise_col
        self.hypothesis_col = hypothesis_col
        self.label_col = label_col
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        prem = row[self.premise_col]
        hyp = row[self.hypothesis_col]
        label = int(row[self.label_col])

        enc = self.tokenizer(
            prem,
            hyp,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )

        enc["labels"] = label
        return enc


class NLIBiEncoderDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, premise_col: str, hypothesis_col: str, label_col: str, max_length: int):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.premise_col = premise_col
        self.hypothesis_col = hypothesis_col
        self.label_col = label_col
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        prem = row[self.premise_col]
        hyp = row[self.hypothesis_col]
        label = int(row[self.label_col])

        prem_enc = self.tokenizer(
            prem,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )

        hyp_enc = self.tokenizer(
            hyp,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )

        return {
            "premise_input_ids": prem_enc["input_ids"],
            "premise_attention_mask": prem_enc["attention_mask"],
            "hypothesis_input_ids": hyp_enc["input_ids"],
            "hypothesis_attention_mask": hyp_enc["attention_mask"],
            "labels": label
        }

In [8]:
# dataloaders

def bi_collate_fn(features):
    premise_features = [
        {
            "input_ids": f["premise_input_ids"],
            "attention_mask": f["premise_attention_mask"]
        }
        for f in features
    ]

    hypothesis_features = [
        {
            "input_ids": f["hypothesis_input_ids"],
            "attention_mask": f["hypothesis_attention_mask"]
        }
        for f in features
    ]

    premise_batch = tokenizer.pad(
        premise_features,
        padding=True,
        return_tensors="pt"
    )

    hypothesis_batch = tokenizer.pad(
        hypothesis_features,
        padding=True,
        return_tensors="pt"
    )

    labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)

    batch = {
        "premise_input_ids": premise_batch["input_ids"],
        "premise_attention_mask": premise_batch["attention_mask"],
        "hypothesis_input_ids": hypothesis_batch["input_ids"],
        "hypothesis_attention_mask": hypothesis_batch["attention_mask"],
        "labels": labels
    }
    return batch


train_ds = NLIPairDataset(
    train_df, tokenizer,
    premise_col=premise_col,
    hypothesis_col=hypothesis_col,
    label_col="label_id",
    max_length=MAX_LENGTH
)
dev_ds = NLIPairDataset(
    dev_df, tokenizer,
    premise_col=premise_col,
    hypothesis_col=hypothesis_col,
    label_col="label_id",
    max_length=MAX_LENGTH
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=data_collator
)
dev_loader = DataLoader(
    dev_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=data_collator
)

train_bi_ds = NLIBiEncoderDataset(
    train_df, tokenizer,
    premise_col=premise_col,
    hypothesis_col=hypothesis_col,
    label_col="label_id",
    max_length=MAX_LENGTH
)
dev_bi_ds = NLIBiEncoderDataset(
    dev_df, tokenizer,
    premise_col=premise_col,
    hypothesis_col=hypothesis_col,
    label_col="label_id",
    max_length=MAX_LENGTH
)

train_bi_loader = DataLoader(
    train_bi_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=bi_collate_fn
)
dev_bi_loader = DataLoader(
    dev_bi_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=bi_collate_fn
)

print("Cross train/dev batches:", len(train_loader), len(dev_loader))
print("Bi train/dev batches:", len(train_bi_loader), len(dev_bi_loader))

Cross train/dev batches: 1527 421
Bi train/dev batches: 1527 421


## Model Definitions

This section defines the two model architectures used in the system.

- The **cross-encoder** uses RoBERTa to jointly model the premise and hypothesis, followed by a custom classifier head.
- The **bi-encoder** uses a shared RoBERTa encoder for both texts, applies mean pooling, and then compares the two sentence representations using concatenation and absolute difference.

In [9]:

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


class RobertaCustomNLI(nn.Module):
    def __init__(self, model_name: str, num_labels: int, dropout: float = 0.2, hidden_dim_1: int = 256, hidden_dim_2: int = 128):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_dim_1),
            nn.LayerNorm(hidden_dim_1),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.LayerNorm(hidden_dim_2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim_2, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_repr)
        return logits


class RobertaBiEncoderNLI(nn.Module):
    def __init__(self, model_name: str, num_labels: int, dropout: float = 0.2, hidden_dim_1: int = 256, hidden_dim_2: int = 128):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        combined_size = hidden_size * 3

        self.classifier = nn.Sequential(
            nn.Linear(combined_size, hidden_dim_1),
            nn.LayerNorm(hidden_dim_1),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.LayerNorm(hidden_dim_2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim_2, num_labels)
        )

    def forward(
        self,
        premise_input_ids,
        premise_attention_mask,
        hypothesis_input_ids,
        hypothesis_attention_mask
    ):
        premise_outputs = self.encoder(
            input_ids=premise_input_ids,
            attention_mask=premise_attention_mask
        )
        hypothesis_outputs = self.encoder(
            input_ids=hypothesis_input_ids,
            attention_mask=hypothesis_attention_mask
        )

        u = mean_pooling(premise_outputs.last_hidden_state, premise_attention_mask)
        v = mean_pooling(hypothesis_outputs.last_hidden_state, hypothesis_attention_mask)

        features = torch.cat([u, v, torch.abs(u - v)], dim=-1)
        logits = self.classifier(features)
        return logits

In [10]:
# evaluation and training helpers

@torch.no_grad()
def evaluate(model, dataloader, criterion):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    for batch in tqdm(dataloader, desc="Evaluating", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        total_loss += loss.item()

        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(dataloader))
    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, f1_macro, all_labels, all_preds


@torch.no_grad()
def evaluate_bi(model, dataloader, criterion):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    for batch in tqdm(dataloader, desc="Evaluating Bi-Encoder", leave=False):
        premise_input_ids = batch["premise_input_ids"].to(device)
        premise_attention_mask = batch["premise_attention_mask"].to(device)
        hypothesis_input_ids = batch["hypothesis_input_ids"].to(device)
        hypothesis_attention_mask = batch["hypothesis_attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(
            premise_input_ids=premise_input_ids,
            premise_attention_mask=premise_attention_mask,
            hypothesis_input_ids=hypothesis_input_ids,
            hypothesis_attention_mask=hypothesis_attention_mask
        )

        loss = criterion(logits, labels)
        total_loss += loss.item()

        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(dataloader))
    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, f1_macro, all_labels, all_preds


def train_one_epoch(model, dataloader, optimizer, scheduler, criterion, epoch_idx: int):
    model.train()
    running_loss = 0.0

    for batch in tqdm(dataloader, desc=f"Training epoch {epoch_idx+1}", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()

    return running_loss / max(1, len(dataloader))


def train_one_epoch_bi(model, dataloader, optimizer, scheduler, criterion, epoch_idx: int):
    model.train()
    running_loss = 0.0

    for batch in tqdm(dataloader, desc=f"Training Bi-Encoder epoch {epoch_idx+1}", leave=False):
        premise_input_ids = batch["premise_input_ids"].to(device)
        premise_attention_mask = batch["premise_attention_mask"].to(device)
        hypothesis_input_ids = batch["hypothesis_input_ids"].to(device)
        hypothesis_attention_mask = batch["hypothesis_attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            premise_input_ids=premise_input_ids,
            premise_attention_mask=premise_attention_mask,
            hypothesis_input_ids=hypothesis_input_ids,
            hypothesis_attention_mask=hypothesis_attention_mask
        )

        loss = criterion(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()

    return running_loss / max(1, len(dataloader))

In [11]:
# train cross-encoder

model = RobertaCustomNLI(
    model_name=MODEL_NAME,
    num_labels=num_labels,
    dropout=DROPOUT,
    hidden_dim_1=HIDDEN_DIM_1,
    hidden_dim_2=HIDDEN_DIM_2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

best_f1 = -1.0
best_acc = -1.0
history = []

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, epoch)

    dev_loss, dev_acc, dev_f1, _, _ = evaluate(model, dev_loader, criterion)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Dev Loss: {dev_loss:.4f} | "
        f"Dev Acc: {dev_acc:.4f} | "
        f"Dev Macro-F1: {dev_f1:.4f}"
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "dev_loss": dev_loss,
        "dev_acc": dev_acc,
        "dev_macro_f1": dev_f1
    })

    if dev_f1 > best_f1:
        best_f1 = dev_f1
        best_acc = dev_acc

        # Save model
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "cross_encoder_model.pt"))

        # Save tokenizer
        tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "cross_encoder_tokenizer"))

        # Save history
        with open(os.path.join(OUTPUT_DIR, "cross_encoder_history.json"), "w") as f:
            json.dump(history, f, indent=2)

        # Save metadata
        metadata = {
            "model_name": MODEL_NAME,
            "num_labels": num_labels,
            "max_length": MAX_LENGTH,
            "batch_size": BATCH_SIZE,
            "epochs_trained": epoch + 1,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "warmup_ratio": WARMUP_RATIO,
            "max_grad_norm": MAX_GRAD_NORM,
            "dropout": DROPOUT,
            "hidden_dim_1": HIDDEN_DIM_1,
            "hidden_dim_2": HIDDEN_DIM_2,
            "best_dev_acc": best_acc,
            "best_dev_macro_f1": best_f1,
            "label2id": label2id,
            "id2label": id2label,
            "premise_col": premise_col,
            "hypothesis_col": hypothesis_col,
            "label_col": label_col
        }

        with open(os.path.join(OUTPUT_DIR, "cross_encoder_metadata.json"), "w") as f:
            json.dump(metadata, f, indent=2)


print("\nBest Cross Dev Acc:", best_acc)
print("Best Cross Dev Macro-F1:", best_f1)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training epoch 1:   0%|          | 0/1527 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/421 [00:00<?, ?it/s]

Epoch 1/2 | Train Loss: 0.4365 | Dev Loss: 0.3391 | Dev Acc: 0.8697 | Dev Macro-F1: 0.8697


Training epoch 2:   0%|          | 0/1527 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/421 [00:00<?, ?it/s]

Epoch 2/2 | Train Loss: 0.2392 | Dev Loss: 0.3609 | Dev Acc: 0.8806 | Dev Macro-F1: 0.8804

Best Cross Dev Acc: 0.8806413301662708
Best Cross Dev Macro-F1: 0.8804406069029829


In [ ]:
# train bi-encoder

bi_model = RobertaBiEncoderNLI(
    model_name=MODEL_NAME,
    num_labels=num_labels,
    dropout=DROPOUT,
    hidden_dim_1=HIDDEN_DIM_1,
    hidden_dim_2=HIDDEN_DIM_2
).to(device)

bi_criterion = nn.CrossEntropyLoss()
bi_optimizer = AdamW(bi_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

bi_total_steps = len(train_bi_loader) * EPOCHS
bi_warmup_steps = int(bi_total_steps * WARMUP_RATIO)

bi_scheduler = get_linear_schedule_with_warmup(
    bi_optimizer,
    num_warmup_steps=bi_warmup_steps,
    num_training_steps=bi_total_steps
)

best_bi_f1 = -1.0
best_bi_acc = -1.0
bi_history = []

for epoch in range(EPOCHS):
    bi_train_loss = train_one_epoch_bi(
        bi_model, train_bi_loader, bi_optimizer, bi_scheduler, bi_criterion, epoch
    )

    bi_dev_loss, bi_dev_acc, bi_dev_f1, _, _ = evaluate_bi(
        bi_model, dev_bi_loader, bi_criterion
    )

    print(
        f"[Bi-Encoder] Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {bi_train_loss:.4f} | "
        f"Dev Loss: {bi_dev_loss:.4f} | "
        f"Dev Acc: {bi_dev_acc:.4f} | "
        f"Dev Macro-F1: {bi_dev_f1:.4f}"
    )

    bi_history.append({
        "epoch": epoch + 1,
        "train_loss": bi_train_loss,
        "dev_loss": bi_dev_loss,
        "dev_acc": bi_dev_acc,
        "dev_macro_f1": bi_dev_f1
    })

    if bi_dev_f1 > best_bi_f1:
        best_bi_f1 = bi_dev_f1
        best_bi_acc = bi_dev_acc

        # Save model
        torch.save(bi_model.state_dict(), os.path.join(BI_OUTPUT_DIR, "bi_encoder_model.pt"))

        # Save tokenizer
        tokenizer.save_pretrained(os.path.join(BI_OUTPUT_DIR, "bi_encoder_tokenizer"))

        # Save history
        with open(os.path.join(BI_OUTPUT_DIR, "bi_encoder_history.json"), "w") as f:
            json.dump(bi_history, f, indent=2)

        # Save metadata
        bi_metadata = {
            "model_name": MODEL_NAME,
            "num_labels": num_labels,
            "max_length": MAX_LENGTH,
            "batch_size": BATCH_SIZE,
            "epochs_trained": epoch + 1,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "warmup_ratio": WARMUP_RATIO,
            "max_grad_norm": MAX_GRAD_NORM,
            "dropout": DROPOUT,
            "hidden_dim_1": HIDDEN_DIM_1,
            "hidden_dim_2": HIDDEN_DIM_2,
            "best_dev_acc": best_bi_acc,
            "best_dev_macro_f1": best_bi_f1
        }

        with open(os.path.join(BI_OUTPUT_DIR, "bi_encoder_metadata.json"), "w") as f:
            json.dump(bi_metadata, f, indent=2)


print("\nBest Bi Dev Acc:", best_bi_acc)
print("Best Bi Dev Macro-F1:", best_bi_f1)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training Bi-Encoder epoch 1:   0%|          | 0/1527 [00:00<?, ?it/s]

Evaluating Bi-Encoder:   0%|          | 0/421 [00:00<?, ?it/s]

[Bi-Encoder] Epoch 1/2 | Train Loss: 0.5550 | Dev Loss: 0.4826 | Dev Acc: 0.7715 | Dev Macro-F1: 0.7706


Training Bi-Encoder epoch 2:   0%|          | 0/1527 [00:00<?, ?it/s]

Evaluating Bi-Encoder:   0%|          | 0/421 [00:00<?, ?it/s]

[Bi-Encoder] Epoch 2/2 | Train Loss: 0.4118 | Dev Loss: 0.4748 | Dev Acc: 0.7904 | Dev Macro-F1: 0.7892

Best Bi Dev Acc: 0.7903800475059383
Best Bi Dev Macro-F1: 0.7892465398587094
